In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList, save_generated_outputs
from resources.prompt_scenarios import prompts_en

# Loading Data, Model and Steering Vectors 
We extract the first 200 examples of each emotion from each languange 

## Indonesian and English Text Dataset

In [2]:
# English Data Load 
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=400, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")
# Indonesian Data Load 
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=400,emotion_dir="resources/id_emotion")

# For steering extraction, we will use the first 200 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.
indo_emotion ={
    "anger": anger_statement_ID[:200],
    "happiness": happiness_statement_ID[:200],
    "sadness": sadness_statement_ID[:200],
    "neutral": neutral_statement_ID[:200],
    "fear": fear_statement_ID[:200],
    "love": love_statement_ID[:200]
}
eng_emotion ={
    "anger": anger_statement[:200],
    "happiness": happiness_statement[:200],
    "sadness": sadness_statement[:200],
    "neutral": neutral_statement[:200],
    "fear": fear_statement[:200],
    "love": love_statement[:200]
}

# For probing, and hidden state analysis we will use all 400
indo_emotion_probe ={
    "anger": anger_statement_ID[:400],
    "happiness": happiness_statement_ID[:400],
    "sadness": sadness_statement_ID[:400],
    "neutral": neutral_statement_ID[:400],
    "fear": fear_statement_ID[:400],
    "love": love_statement_ID[:400]
}
eng_emotion_probe ={
    "anger": anger_statement[:400],
    "happiness": happiness_statement[:400],
    "sadness": sadness_statement[:400],
    "neutral": neutral_statement[:400],
    "fear": fear_statement[:400],
    "love": love_statement[:400]
}

In [ ]:
# Indoensian Data Sample
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

# English Data Sample 
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

## Model Loading 

In [4]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.6P  732T  69% /workspace


In [4]:
model,tokenizer = setup.modelSetup()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

## Extracting Probing Data and Steering Vector 
Skip this step if you have steering vector already loaded, or ran this before. 

In [ ]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion_probe, name_folder="English Vectors", only_return_emotion_vectors=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion_probe, name_folder="Indonesian Vectors", only_return_emotion_vectors=True)

In [ ]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

Run this if you have already ran the code above beforehand

In [6]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

# Steering Response Analysis 
Here we run the LLMs against a list of nuetral prompts

In [8]:
import resources.anger_prompt.prompt_scenarios_spectrum as resource_spectrum
import resources.anger_prompt.prompt_scenarios_cultural as resource_cultural
import resources.fear_prompt.prompt_scenarios_cultural as resource_cultural_fear
import resources.neutral_prompts.prompt_neutral as resource_neutral

def force_reload_prompt_modules():
    importlib.invalidate_caches()
    for module in (resource_spectrum, resource_cultural, resource_cultural_fear, resource_neutral):
        importlib.reload(module)

force_reload_prompt_modules()

# neutral
prompts_id_neutral = resource_neutral.prompt_neutral_id_1


# Load and normalize steering vectors used by all scenario blocks
# steering_vector_english = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
# steering_vector_indo = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")
steering_vector_eng = steering_vector_eng
steering_vector_id = steering_vector_id
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# Indo nomalisation
# steering_vectors_lang_id = norm_vectors(steering_vectors_lang_id)




## Model System Prompts and Settings

In [9]:
system_prompt_reaction_id = """
Kamu adalah chatbot yang membantu.
Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal
Maksimal 60 kata.
"""

In [40]:
# list_steering_strengths = [0.15,0.2,0.3]
list_steering_strengths = [ 1.5, 2,2.5] 
# Commong Settings
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_neutral[:5],
    "target_layers": [18, 19, 20],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
}

## Running the LLMs 

### Indonesian Steer 

In [41]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
texts_generated_neutral_anger_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

texts_generated_neutral_fear_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

texts_generated_neutral_happiness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

texts_generated_neutral_sadness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

texts_generated_neutral_love_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )

texts_generated_neutral_id = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

Scenario List Neutral (Indonesian fear vector): 100%|██████████| 15/15 [01:59<00:00,  8.00s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 15/15 [02:04<00:00,  8.32s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 15/15 [02:03<00:00,  8.22s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 15/15 [02:05<00:00,  8.36s/it]
Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 15/15 [02:08<00:00,  8.60s/it]


In [43]:
save_generated_outputs({
    k: v for k, v in globals().items() if k.startswith('texts_generated_')},
    output_path='outputs/good_5_texts.json'
    )

'outputs/good_5_texts.json'

In [42]:
# Steering Response analysis Neutral (Indonesian only, five emotion vectors)
required_id = [
    'texts_generated_neutral_anger_id',
    'texts_generated_neutral_fear_id',
    'texts_generated_neutral_happiness_id',
    'texts_generated_neutral_sadness_id',
    'texts_generated_neutral_love_id',
    'texts_generated_neutral_id',
]

missing_id = [name for name in required_id if name not in globals()]
if missing_id:
    print('No output to print yet. Run the Indonesian generation cell first.')
    print('Missing variables:', ', '.join(missing_id))
elif not texts_generated_neutral_anger_id:
    print('No output to print: texts_generated_neutral_anger_id is empty.')
else:
    print(f"Total prompts to print: {len(texts_generated_neutral_anger_id)}")
    for prompt in texts_generated_neutral_anger_id:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("Indonesian anger vector")
        for result in texts_generated_neutral_anger_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian fear vector")
        for result in texts_generated_neutral_fear_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian happiness vector")
        for result in texts_generated_neutral_happiness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian sadness vector")
        for result in texts_generated_neutral_sadness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian love vector")
        for result in texts_generated_neutral_love_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian Neutral vector")
        for result in texts_generated_neutral_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

Total prompts to print: 5
Prompt: Jelaskan bagaimana seseorang bereaksi terhadap suatu berita.
----------------------------------------
Indonesian anger vector
Steering Strength: 1.5
Generated Text: Seseorang mungkin bereaksi dengan emosi yang kuat terhadap berita yang menimbulkan perubahan besar dalam hidupnya. Mereka mungkin merasa terkejut, takut, atau bahkan sedih. Mereka mungkin merenungkan dampak berita tersebut terhadap masa depan mereka.
------------
Steering Strength: 2
Generated Text: Seseorang dapat bereaksi dengan emosi, misalnya:

*   Kebijaksanaan: Mereka dapat berpikir dengan tenang dan matang.
*   Kekecewaan: Mereka dapat merasa marah atau kecewa.
*   Kegembiraan: Mereka dapat merasa senang atau bahagia.
*   Keterkejutan: Mereka dapat terkejut atau terpikir.
------------
Steering Strength: 2.5
Generated Text: Wah, kalau saya mendengar suatu berita, saya akan mengalami reaksi emosional. Saya mungkin akan terkejut, marah, sedih, atau bahkan terkecundal. Reaksi saya akan b

### English Steering 

In [ ]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
texts_generated_neutral_anger_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['anger'],
    progress_desc="Scenario List Neutral (English anger vector)"
 )

texts_generated_neutral_fear_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['fear'],
    progress_desc="Scenario List Neutral (English fear vector)"
 )

texts_generated_neutral_happiness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['happiness'],
    progress_desc="Scenario List Neutral (English happiness vector)"
 )

texts_generated_neutral_sadness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['sadness'],
    progress_desc="Scenario List Neutral (English sadness vector)"
 )

texts_generated_neutral_love_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['love'],
    progress_desc="Scenario List Neutral (English love vector)"
 )

In [29]:
# Steering Response analysis Neutral (English only, five emotion vectors)
required_eng = [
    'texts_generated_neutral_anger_eng',
    'texts_generated_neutral_fear_eng',
    'texts_generated_neutral_happiness_eng',
    'texts_generated_neutral_sadness_eng',
    'texts_generated_neutral_love_eng',
]

missing_eng = [name for name in required_eng if name not in globals()]
if missing_eng:
    print('No output to print yet. Run the English generation cell first.')
    print('Missing variables:', ', '.join(missing_eng))
elif not texts_generated_neutral_anger_eng:
    print('No output to print: texts_generated_neutral_anger_eng is empty.')
else:
    print(f"Total prompts to print: {len(texts_generated_neutral_anger_eng)}")
    for prompt in texts_generated_neutral_anger_eng:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("English anger vector")
        for result in texts_generated_neutral_anger_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English fear vector")
        for result in texts_generated_neutral_fear_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English happiness vector")
        for result in texts_generated_neutral_happiness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English sadness vector")
        for result in texts_generated_neutral_sadness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English love vector")
        for result in texts_generated_neutral_love_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

No output to print yet. Run the English generation cell first.
Missing variables: texts_generated_neutral_anger_eng, texts_generated_neutral_fear_eng, texts_generated_neutral_happiness_eng, texts_generated_neutral_sadness_eng, texts_generated_neutral_love_eng
